In [1]:
import os
import sys

# 1. Force the notebook to run from your project workspace root
project_root = r"C:\Users\Greesha Vaishnavi\Desktop\dsprojects\linear_regression"
os.chdir(project_root)

# 2. Add the 'src' folder to Python's system path so it can see 'Linear_regression_01'
src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

print("Current Working Directory:", os.getcwd())
print("System path updated. Available modules:", os.listdir(src_path))

Current Working Directory: C:\Users\Greesha Vaishnavi\Desktop\dsprojects\linear_regression
System path updated. Available modules: ['linear_regression', 'Linear_regression_01', 'Linear_regression_01.egg-info']


In [2]:
import os
print(os.getcwd())

C:\Users\Greesha Vaishnavi\Desktop\dsprojects\linear_regression


In [3]:
import sys
print(sys.path)

['c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\python310.zip', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\DLLs', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\lib', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire', '', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\lib\\site-packages', 'C:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\linear_regression\\src', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\lib\\site-packages\\win32', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\lib\\site-packages\\win32\\lib', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\lib\\site-packages\\pythonwin']


In [4]:
import os

print(os.path.exists("../src"))
print(os.path.exists("../src/Linear_regression_01"))

False
False


In [5]:
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
SRC_PATH = os.path.join(PROJECT_ROOT, "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

print(PROJECT_ROOT)
print(SRC_PATH)
print(sys.path[:3])

C:\Users\Greesha Vaishnavi\Desktop\dsprojects
C:\Users\Greesha Vaishnavi\Desktop\dsprojects\src
['C:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\src', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\python310.zip', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\DLLs']


In [ ]:
import box
print(box.__version__) # to manage complex hyperparameter configurations and clean up model architectures

7.4.1


In [7]:
# 3 entites update
# from config.ymal
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen = True)
class DataIngestionConfig:
    root_dir: Path
    dataset_name: str
    local_data_file: Path
    unzip_dir: Path

In [8]:
from Linear_regression_01.constant import *
from Linear_regression_01.utils.common import read_yaml, create_directories

In [9]:
# 4 Update configuration manager

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,     # Access to constants
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath) # read all config and params yaml files
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:

        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            dataset_name=config.dataset_name,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

        return data_ingestion_config

In [10]:
import os
import zipfile
from pathlib import Path
from Linear_regression_01.logging import logger
from Linear_regression_01.utils.common import get_size

In [ ]:
# 5 Components

import os
import zipfile
from kaggle.api.kaggle_api_extended import KaggleApi

from Linear_regression_01.logging import logger


class DataIngestion:

    def __init__(self, config):
        self.config = config

    def download_files(self):

        if not os.path.exists(self.config.local_data_file):  #download file from kaggle

            api = KaggleApi()
            api.authenticate() # from kaggle.json

            api.dataset_download_files(
                dataset=self.config.dataset_name,
                path=self.config.root_dir,
                unzip=False # Because Download Extraction are separate responsibilities.
            )

            logger.info("Dataset downloaded successfully")

        else:
            logger.info("Dataset already exists")

    def extract_zip_file(self):

        os.makedirs(self.config.unzip_dir, exist_ok=True)

        with zipfile.ZipFile(self.config.local_data_file, "r") as zip_ref:
            zip_ref.extractall(self.config.unzip_dir)

        logger.info("Dataset extracted successfully")


        

In [12]:
# 6 pipeline
import zipfile
import os
try:
    print("--- STARTING FULL PIPELINE RUN ---")
    config = ConfigurationManager()

    data_ingestion_config = config.get_data_ingestion_config()
    
    pipeline_component = DataIngestion(config=data_ingestion_config)
    
    # 1. This will say "File already exists!" since we just downloaded it
    pipeline_component.download_files()
    
    # 2. This will unpack your dataset
    pipeline_component.extract_zip_file()
    
    print("--- ALL SANDBOX STEPS COMPLETED SUCCESSFULLY ---")

except Exception as e:
    raise e

--- STARTING FULL PIPELINE RUN ---
[2026-07-22 20:38:10,862: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-22 20:38:10,873: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-22 20:38:10,873: INFO: common: created directory at artifacts]
[2026-07-22 20:38:10,877: INFO: common: created directory at artifacts/data_ingestion]
[2026-07-22 20:38:10,877: INFO: 1924905181: Dataset already exists]
[2026-07-22 20:38:10,890: INFO: 1924905181: Dataset extracted successfully]
--- ALL SANDBOX STEPS COMPLETED SUCCESSFULLY ---
